### **Setup**

In [15]:
import os
from dotenv import load_dotenv

load_dotenv()

try:
    import langgraph_supervisor
    print("langgraph-supervisor: installed")
    SUPERVISOR_AVAILABLE = True
except ImportError:
    print("WARNING: langgraph-supervisor not installed. Run: uv pip install langgraph-supervisor>=0.1.0")
    SUPERVISOR_AVAILABLE = False

try:
    import langgraph_swarm
    print("langgraph-swarm: installed")
    SWARM_AVAILABLE = True
except ImportError:
    print("WARNING: langgraph-swarm not installed. Run: uv pip install langgraph-swarm>=0.1.0")
    SWARM_AVAILABLE = False

print("Setup complete")

langgraph-supervisor: installed
langgraph-swarm: installed
Setup complete


### **Custom Coordinator Pattern (No Extra Packages)**

In [4]:
import operator
import json
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# ── State ────────────────────────────────────────────────────────────────────
class CoordinatorState(TypedDict):
    task: str                                         # Original user request
    messages: Annotated[list, operator.add]           # Full conversation log
    research_notes: Annotated[list[str], operator.add]  # Accumulated by researcher
    analysis_notes: Annotated[list[str], operator.add]  # Accumulated by analyst
    draft: str                                        # Written by writer (replaces)
    routing_history: Annotated[list[str], operator.add]  # Audit trail
    step_count: int

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)

In [5]:

# ── Coordinator node ─────────────────────────────────────────────────────────
ROUTING_SYSTEM = """
You are a coordinator managing a 3-person research team.
Given the current task and what has been done, decide the next step.

Available agents:
- researcher: gathers facts and information
- analyst: interprets findings and draws conclusions
- writer: produces the final written output
- FINISH: task is complete — return this when the writer has produced a draft

Rules:
1. Always research before analysing.
2. Always analyse before writing.
3. Return FINISH once the writer has produced output.
4. Never exceed 5 total steps.

Reply with ONLY one of: researcher | analyst | writer | FINISH
"""

def coordinator(state: CoordinatorState) -> Command:
    """LLM-driven coordinator — decides which agent to call next."""
    history_summary = ", ".join(state["routing_history"]) or "none yet"
    
    prompt = (
        f"Task: {state['task']}\n"
        f"Steps taken: {history_summary}\n"
        f"Research notes: {'yes' if state['research_notes'] else 'none'}\n"
        f"Analysis notes: {'yes' if state['analysis_notes'] else 'none'}\n"
        f"Draft produced: {'yes' if state['draft'] else 'no'}\n"
        f"Total steps so far: {state['step_count']}"
    )
    
    response = llm.invoke([
        SystemMessage(ROUTING_SYSTEM),
        HumanMessage(prompt),
    ])
    
    decision = response.content.strip().lower()
    print(f"[Coordinator] Step {state['step_count'] + 1} → {decision}")

    valid = {"researcher", "analyst", "writer", "finish"}
    if decision not in valid:
        decision = "finish"
    
    next_node = END if decision == "finish" else decision
    
    return Command(
        update={
            "routing_history": [decision],
            "step_count": state["step_count"] + 1,
        },
        goto=next_node,
    )


In [6]:

# ── Specialist nodes ──────────────────────────────────────────────────────────
def researcher(state: CoordinatorState) -> Command:
    """Gather facts — returns research notes and routes back to coordinator."""
    response = llm.invoke([
        SystemMessage("You are a researcher. Provide 3 concise bullet-point facts."),
        HumanMessage(f"Research: {state['task']}"),
    ])
    note = response.content
    print(f"[Researcher] Note: {note[:70]}...")
    return Command(
        update={"research_notes": [note]},
        goto="coordinator",
    )

def analyst(state: CoordinatorState) -> Command:
    """Analyse research notes — routes back to coordinator."""
    combined = "\n".join(state["research_notes"])
    response = llm.invoke([
        SystemMessage("You are an analyst. Provide 2 key insights from the research."),
        HumanMessage(f"Research:\n{combined}\n\nTask: {state['task']}"),
    ])
    insight = response.content
    print(f"[Analyst] Insight: {insight[:70]}...")
    return Command(
        update={"analysis_notes": [insight]},
        goto="coordinator",
    )

def writer(state: CoordinatorState) -> Command:
    """Produce a written summary — routes back to coordinator."""
    research = "\n".join(state["research_notes"])
    analysis = "\n".join(state["analysis_notes"])
    response = llm.invoke([
        SystemMessage("You are a technical writer. Write a clear 3-sentence executive summary."),
        HumanMessage(
            f"Task: {state['task']}\n\nResearch:\n{research}\n\nAnalysis:\n{analysis}"
        ),
    ])
    draft = response.content
    print(f"[Writer] Draft produced ({len(draft)} chars)")
    return Command(
        update={"draft": draft},
        goto="coordinator",
    )

In [7]:
# ── Build graph ────────────────────────────────────────────────────────────────
builder = StateGraph(CoordinatorState)
builder.add_node("coordinator", coordinator)
builder.add_node("researcher", researcher)
builder.add_node("analyst", analyst)
builder.add_node("writer", writer)

builder.add_edge(START, "coordinator")

custom_coordinator_app = builder.compile()
print("Custom coordinator graph compiled")

Custom coordinator graph compiled


In [8]:
# Run the custom coordinator
print("=" * 60)
print("Custom Coordinator — Multi-Agent Research")
print("=" * 60)

result = custom_coordinator_app.invoke({
    "task": "The impact of LangGraph on enterprise AI adoption in 2025",
    "messages": [],
    "research_notes": [],
    "analysis_notes": [],
    "draft": "",
    "routing_history": [],
    "step_count": 0,
})

print()
print("Routing path:", " -> ".join(result["routing_history"]))
print()
print("Final Draft:")
print("-" * 60)
print(result["draft"])

Custom Coordinator — Multi-Agent Research
[Coordinator] Step 1 → researcher
[Researcher] Note: - **Accelerated AI Integration:** LangGraph, a next-generation languag...
[Coordinator] Step 2 → analyst
[Analyst] Insight: 1. **Accelerated AI Deployment and Interoperability:** LangGraph is se...
[Coordinator] Step 3 → writer
[Writer] Draft produced (553 chars)
[Coordinator] Step 4 → finish

Routing path: researcher -> analyst -> writer -> finish

Final Draft:
------------------------------------------------------------
LangGraph is poised to transform enterprise AI adoption in 2025 by streamlining deployment timelines and enabling seamless interoperability across diverse systems, addressing a critical challenge in AI integration. Its ability to unify structured and unstructured data enhances data accessibility and accuracy, driving advanced predictive analytics and more informed decision-making. Additionally, LangGraph's cost-efficient, modular architecture supports scalable AI solutions, 

### **Supervisor Pattern (langgraph-supervisor)**

In [10]:
from langchain_core.tools import tool
# ── Simulated domain tools ────────────────────────────────────────────────────
@tool
def search_papers(query: str) -> str:
    """Search academic papers and return key findings."""
    return (
        f"Found 3 papers on '{query}':\n"
        "  1. 'Stateful LLM Workflows' (2025) — Key finding: graph-based state machines "
        "reduce hallucinations by 34%.\n"
        "  2. 'Agent Coordination at Scale' (2025) — Supervisor patterns outperform monolithic agents.\n"
        "  3. 'Enterprise AI Adoption' (2026) — 78% of Fortune 500 pilots use agentic frameworks."
    )

@tool
def analyze_trends(topic: str, data_points: str) -> str:
    """Analyze a set of data points and return trend insights."""
    return (
        f"Trend analysis for '{topic}' based on: {data_points[:80]}...\n"
        "  - Growth rate: 45% YoY\n"
        "  - Primary driver: cost reduction vs. human labor\n"
        "  - Risk: governance and compliance concerns remain top barrier"
    )

@tool
def write_executive_summary(title: str, findings: str) -> str:
    """Write a professional executive summary from research findings."""
    return (
        f"# {title}\n\n"
        f"**Executive Summary**\n\n"
        f"Based on current research, {findings[:120]}... "
        "The data indicates strong momentum with measurable ROI in early adopter organizations. "
        "Recommended action: begin a structured proof-of-concept within 90 days."
    )

print("Domain tools defined: search_papers, analyze_trends, write_executive_summary")

Domain tools defined: search_papers, analyze_trends, write_executive_summary


In [12]:
if not SUPERVISOR_AVAILABLE:
    print("SKIPPED — install langgraph-supervisor to run this section")
    print("  uv pip install langgraph-supervisor>=0.1.0")
else:
    from langgraph_supervisor import create_supervisor
    from langgraph.checkpoint.memory import MemorySaver

    llm_mini = AzureChatOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        azure_deployment="gpt-4o-mini",
        temperature=0.7,
    )
    llm_smart = AzureChatOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        azure_deployment="gpt-4o",
        temperature=0.7,
    )
    from langgraph.prebuilt import create_react_agent

    research_agent = create_react_agent(
        llm_mini,
        tools=[search_papers],
        prompt=(
            "You are a research specialist. Use search_papers to find information. "
            "Always return structured findings with numbered points."
        ),
        name="researcher",
    )

    analysis_agent = create_react_agent(
        llm_mini,
        tools=[analyze_trends],
        prompt=(
            "You are a data analyst. Use analyze_trends to provide quantified insights. "
            "Focus on numbers, growth rates, and risk factors."
        ),
        name="analyst",
    )

    writing_agent = create_react_agent(
        llm_smart,
        tools=[write_executive_summary],
        prompt=(
            "You are a technical writer. Use write_executive_summary to produce "
            "polished, professional output suitable for C-suite executives."
        ),
        name="writer",
    )
    # ── Supervisor ────────────────────────────────────────────────────────────
    supervisor_app = create_supervisor(
        agents=[research_agent, analysis_agent, writing_agent],
        model=llm_smart,
        prompt=(
            "You are the research team supervisor. Coordinate three specialists:\n"
            "  - researcher: gathers academic papers and facts\n"
            "  - analyst: interprets data and identifies trends\n"
            "  - writer: produces executive-quality reports\n\n"
            "Always research first, then analyse, then write. "
            "Synthesize all results into a final response for the user."
        ),
    ).compile(checkpointer=MemorySaver())

    print("Supervisor compiled with 3 sub-agents")

    # ── Run ───────────────────────────────────────────────────────────────────
    config = {"configurable": {"thread_id": "supervisor-demo-001"}}
    print()
    print("=" * 60)
    print("Supervisor Pattern — Research Team")
    print("=" * 60)

    result = supervisor_app.invoke(
        {"messages": [{"role": "user", "content": (
            "Research the impact of LangGraph on enterprise AI adoption. "
            "Search for papers, analyse the trends, then write an executive summary."
        )}]},
        config,
    )

    print("\nFinal supervisor response:")
    print("-" * 60)
    print(result["messages"][-1].content)

C:\Users\mohammed.a\AppData\Local\Temp\ipykernel_35864\1234874975.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(
C:\Users\mohammed.a\AppData\Local\Temp\ipykernel_35864\1234874975.py:34: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  analysis_agent = create_react_agent(
C:\Users\mohammed.a\AppData\Local\Temp\ipykernel_35864\1234874975.py:44: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  writing_agent = create_react_agent(


Supervisor compiled with 3 sub-agents

Supervisor Pattern — Research Team

Final supervisor response:
------------------------------------------------------------
### Final Executive Summary: Impact of LangGraph on Enterprise AI Adoption

LangGraph is revolutionizing enterprise AI adoption by addressing critical challenges and enhancing the efficiency of AI applications. Research reveals that LangGraph has led to:

1. **Reduction in Errors**: A 34% decrease in hallucinations improves reliability, making AI outputs more trustworthy for business-critical applications.
2. **Enhanced Performance**: LangGraph-powered agents outperform traditional models by 15% in complex tasks, enabling more effective operations.
3. **Widespread Adoption**: Pilot projects utilizing LangGraph are active in 78% of Fortune 500 companies, demonstrating significant acceptance and confidence in its capabilities.

The primary driver for its adoption is cost-efficiency, offering a compelling alternative to human la

### **Swarm Pattern (langgraph-swarm)**

In [13]:
from langchain_core.tools import tool
from langchain_openai import AzureChatOpenAI

# ── Domain tools ──────────────────────────────────────────────────────────────
@tool
def check_account_status(account_id: str) -> str:
    """Check account balance, status, and recent transactions."""
    return (
        f"Account {account_id}: Active | Balance: $1,234.56 | "
        "Last transaction: $45.00 on 2026-04-23 | No holds."
    )

@tool
def troubleshoot_api_error(error_code: str, service: str) -> str:
    """Diagnose and resolve API or technical errors."""
    return (
        f"Error {error_code} on {service}: Authentication token expired. "
        "Resolution: regenerate API keys in Settings > Security > API Keys. "
        "This resolves 92% of 401 errors."
    )

@tool
def process_refund_request(order_id: str, reason: str) -> str:
    """Process a billing refund or dispute."""
    return (
        f"Refund for order {order_id} (reason: {reason}): Approved for review. "
        "Estimated credit in 3-5 business days. Reference: REF-88321."
    )

print("Customer support tools defined")

Customer support tools defined


In [14]:
if not SWARM_AVAILABLE:
    print("SKIPPED — install langgraph-swarm to run this section")
    print("  uv pip install langgraph-swarm>=0.1.0")
else:
    from langgraph_swarm import create_swarm
    from langgraph.prebuilt import create_react_agent
    from langgraph.checkpoint.memory import MemorySaver

    account_agent = create_react_agent(
        llm_mini,
        tools=[check_account_status],
        prompt=(
            "You are an account management specialist. Handle account inquiries. "
            "If the user has a technical/API issue, transfer to the technical agent. "
            "If the user has a billing dispute, transfer to the billing agent."
        ),
        name="account_agent",
    )

    technical_agent = create_react_agent(
        llm_mini,
        tools=[troubleshoot_api_error],
        prompt=(
            "You are a technical support engineer. Diagnose and resolve API and software errors. "
            "If the user has an account question, transfer to the account agent. "
            "If the user has a billing issue, transfer to the billing agent."
        ),
        name="technical_agent",
    )

    billing_agent = create_react_agent(
        llm_mini,
        tools=[process_refund_request],
        prompt=(
            "You are a billing specialist. Handle refunds, disputes, and payment issues. "
            "If the user has an account question, transfer to the account agent. "
            "If the user has a technical issue, transfer to the technical agent."
        ),
        name="billing_agent",
    )

    # create_swarm wires transfer tools automatically
    support_swarm = create_swarm(
        agents=[account_agent, technical_agent, billing_agent],
        default_active_agent="account_agent",  # First contact point
    ).compile(checkpointer=MemorySaver())

    print("Customer support swarm compiled")
    print("  - account_agent (default entry)")
    print("  - technical_agent")
    print("  - billing_agent")
    print("  Each agent can transfer to either of the other two autonomously.")

Customer support swarm compiled
  - account_agent (default entry)
  - technical_agent
  - billing_agent
  Each agent can transfer to either of the other two autonomously.


C:\Users\mohammed.a\AppData\Local\Temp\ipykernel_35864\2429399393.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  account_agent = create_react_agent(
C:\Users\mohammed.a\AppData\Local\Temp\ipykernel_35864\2429399393.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  technical_agent = create_react_agent(
C:\Users\mohammed.a\AppData\Local\Temp\ipykernel_35864\2429399393.py:31: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  billing_agent = create_react_agent(


In [15]:
if not SWARM_AVAILABLE:
    print("SKIPPED — swarm not available")
else:
    # Test 1: Issue that starts with account agent and routes to technical
    config = {"configurable": {"thread_id": "swarm-demo-001"}}

    print("=" * 60)
    print("Swarm Test 1: Mixed issue (account + technical)")
    print("=" * 60)

    result = support_swarm.invoke(
        {"messages": [{"role": "user", "content": (
            "I cannot log in to my account — I keep getting a 401 API authentication error. "
            "My account ID is ACC-9981."
        )}]},
        config,
    )

    print("\nFinal response:")
    print(result["messages"][-1].content)
    
    # Test 2: Pure billing issue — should stay with billing agent
    config2 = {"configurable": {"thread_id": "swarm-demo-002"}}

    print()
    print("=" * 60)
    print("Swarm Test 2: Billing dispute")
    print("=" * 60)

    result2 = support_swarm.invoke(
        {"messages": [{"role": "user", "content": (
            "I was charged twice for order ORD-5521 last week. "
            "I need a refund for the duplicate charge."
        )}]},
        config2,
    )

    print("\nFinal response:")
    print(result2["messages"][-1].content)

Swarm Test 1: Mixed issue (account + technical)

Final response:
It seems that your account is active, and there are no issues on our end. However, since you're experiencing a 401 API authentication error, I will transfer you to a technical agent who can assist you with this issue. Please hold on for a moment.

Swarm Test 2: Billing dispute

Final response:
I will transfer you to the billing agent to handle your refund request for the duplicate charge on order ORD-5521. Please hold on for a moment.


### **Inter-Agent Communication Patterns**

In [3]:
import os
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

In [4]:
# ── Shared typed state with reducers ──────────────────────────────────────────
class SharedProjectState(TypedDict):
    project_brief: str
    research_findings: Annotated[list[str], operator.add]  # Multiple agents append
    analysis_results: Annotated[list[str], operator.add]
    draft_sections: Annotated[list[str], operator.add]
    final_report: str
    status: str

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)

In [5]:
# ── Agent nodes — each reads prior state and appends its output ───────────────
def research_agent(state: SharedProjectState) -> dict:
    """Research agent — appends to shared research_findings."""
    prompt = f"Provide 2 concise facts about: {state['project_brief']}"
    response = llm.invoke([
        SystemMessage("You are a researcher. Be factual and concise."),
        HumanMessage(prompt),
    ])
    finding = response.content
    print(f"Research agent: {finding[:70]}...")
    return {"research_findings": [finding]}

def analysis_agent(state: SharedProjectState) -> dict:
    """Analyst — reads research_findings, appends analysis_results."""
    findings_text = "\n".join(state["research_findings"])
    prompt = f"Identify 2 key insights from:\n{findings_text}"
    response = llm.invoke([
        SystemMessage("You are an analyst. Be precise."),
        HumanMessage(prompt),
    ])
    analysis = response.content
    print(f"Analysis agent: {analysis[:70]}...")
    return {"analysis_results": [analysis]}

def writing_agent(state: SharedProjectState) -> dict:
    """Writer — reads all prior work, produces final_report."""
    context = (
        f"Project: {state['project_brief']}\n"
        f"Research: {'; '.join(state['research_findings'])}\n"
        f"Analysis: {'; '.join(state['analysis_results'])}"
    )
    prompt = f"Write a 3-sentence executive summary from:\n{context}"
    response = llm.invoke([
        SystemMessage("You are a technical writer."),
        HumanMessage(prompt),
    ])
    report = f"# Project Report: {state['project_brief']}\n\n{response.content}"
    print(f"Writing agent: report generated ({len(report)} chars)")
    return {"final_report": report, "status": "complete"}

In [6]:
# ── Sequential pipeline with shared state ─────────────────────────────────────
builder = StateGraph(SharedProjectState)
builder.add_node("researcher", research_agent)
builder.add_node("analyst", analysis_agent)
builder.add_node("writer", writing_agent)

builder.add_edge(START, "researcher")
builder.add_edge("researcher", "analyst")
builder.add_edge("analyst", "writer")
builder.add_edge("writer", END)

pipeline = builder.compile()

In [7]:
# ── Run ───────────────────────────────────────────────────────────────────────
print("=" * 60)
print("Shared State Pipeline")
print("=" * 60)

result = pipeline.invoke({
    "project_brief": "LangGraph in enterprise AI development — 2026 outlook",
    "research_findings": [],
    "analysis_results": [],
    "draft_sections": [],
    "final_report": "",
    "status": "in_progress",
})

print()
print("=" * 60)
print(result["final_report"])
print(f"\nStatus: {result['status']}")
print(f"Research entries: {len(result['research_findings'])}")
print(f"Analysis entries: {len(result['analysis_results'])}")

Shared State Pipeline
Research agent: 1. **Enhanced Multilingual Capabilities**: By 2026, LangGraph is proje...
Analysis agent: 1. **Global Impact through Multilingual Integration**: By 2026, LangGr...
Writing agent: report generated (618 chars)

# Project Report: LangGraph in enterprise AI development — 2026 outlook

By 2026, LangGraph is poised to revolutionize enterprise AI development through advanced multilingual data integration and natural language processing, fostering global collaboration and expanding market reach. Its custom AI workflow automation capabilities will enable organizations to streamline operations and implement domain-specific solutions, driving efficiency and informed decision-making. As a result, LangGraph is expected to play a pivotal role in enhancing both global impact and operational effectiveness for enterprises worldwide.

Status: complete
Research entries: 1
Analysis entries: 1


### **Error Propagation in Multi-Agent Systems**

In [8]:
import random
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage

class RobustState(TypedDict):
    task: str
    result: str
    error_log: list[str]
    attempts: int
    max_attempts: int

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)

MAX_RETRIES = 3

In [9]:
def unreliable_agent(state: RobustState) -> Command:
    """
    Agent that simulates intermittent failures.
    On the first two attempts it raises; on the third it succeeds.
    In production, failures come from network errors, rate limits, etc.
    """
    attempt = state["attempts"]
    print(f"[Agent] Attempt {attempt + 1}/{state['max_attempts']}")

    try:
        # Simulate failure on first 2 attempts
        if attempt < 2:
            raise RuntimeError(f"Simulated transient error on attempt {attempt + 1}")

        response = llm.invoke([HumanMessage(f"Answer briefly: {state['task']}")])
        print(f"[Agent] Success on attempt {attempt + 1}")
        return Command(
            update={"result": response.content, "attempts": attempt + 1},
            goto=END,
        )

    except Exception as exc:
        error_msg = f"Attempt {attempt + 1}: {exc}"
        print(f"[Agent] FAILED — {error_msg}")


        new_attempts = attempt + 1
        if new_attempts >= state["max_attempts"]:
            # Exhausted retries — degrade gracefully
            print("[Agent] Max retries reached — returning fallback response")
            return Command(
                update={
                    "result": "Service temporarily unavailable. Please try again later.",
                    "error_log": [error_msg],
                    "attempts": new_attempts,
                },
                goto=END,
            )

        # Retry available — route to wait node
        return Command(
            update={"error_log": [error_msg], "attempts": new_attempts},
            goto="retry_handler",
        )

def retry_handler(state: RobustState) -> Command:
    """
    Retry handler — in production you would add exponential backoff here
    (time.sleep(2 ** state['attempts'])) or switch to an alternative service.
    """
    print(f"[Retry] Preparing attempt {state['attempts'] + 1}...")
    # In production: time.sleep(2 ** state['attempts'])
    return Command(goto="agent")

In [10]:
# ── Build graph ────────────────────────────────────────────────────────────────
builder = StateGraph(RobustState)
builder.add_node("agent", unreliable_agent)
builder.add_node("retry_handler", retry_handler)

builder.add_edge(START, "agent")
robust_app = builder.compile()

In [11]:
print("=" * 60)
print("Error Propagation & Retry Demonstration")
print("=" * 60)
print()

result = robust_app.invoke({
    "task": "What is the capital of France?",
    "result": "",
    "error_log": [],
    "attempts": 0,
    "max_attempts": MAX_RETRIES,
})

Error Propagation & Retry Demonstration

[Agent] Attempt 1/3
[Agent] FAILED — Attempt 1: Simulated transient error on attempt 1
[Retry] Preparing attempt 2...
[Agent] Attempt 2/3
[Agent] FAILED — Attempt 2: Simulated transient error on attempt 2
[Retry] Preparing attempt 3...
[Agent] Attempt 3/3
[Agent] Success on attempt 3


In [12]:
print()
print("Final result:", result["result"])
print("Total attempts:", result["attempts"])
if result["error_log"]:
    print("Error log:")
    for entry in result["error_log"]:
        print(f"  - {entry}")


Final result: The capital of France is Paris.
Total attempts: 3
Error log:
  - Attempt 2: Simulated transient error on attempt 2


### **Capstone Project: Production Multi-Agent Research System**

In [13]:
from langchain_core.tools import tool
from langchain_openai import AzureChatOpenAI

# ── Specialist tools ──────────────────────────────────────────────────────────
@tool
def search_technical_papers(query: str) -> str:
    """Search academic and technical papers on a given topic."""
    return (
        f"Technical search for '{query}':\n"
        "  Paper 1: 'Multi-Agent LLM Architectures' (2026) — "
        "Supervisor patterns reduce coordination latency by 28%.\n"
        "  Paper 2: 'LangGraph at Scale' (2025) — "
        "Stateful agents outperform stateless chains for complex tasks.\n"
        "  Paper 3: 'Enterprise AI Benchmarks' (2026) — "
        "Agentic systems show 3.2x better task completion rates."
    )

@tool
def search_industry_reports(query: str) -> str:
    """Search market research and industry analyst reports."""
    return (
        f"Industry reports for '{query}':\n"
        "  Gartner (2026): Agentic AI market $12B, 67% YoY growth.\n"
        "  Forrester (2026): 54% of enterprises running pilot agentic workflows.\n"
        "  McKinsey (2026): ROI of $2.40 per $1 invested in multi-agent systems."
    )

@tool
def synthesize_research(technical_findings: str, industry_findings: str, topic: str) -> str:
    """Synthesize technical and industry research into a coherent analysis."""
    return (
        f"Synthesis for '{topic}':\n\n"
        f"Technical perspective: {technical_findings[:120]}...\n\n"
        f"Market perspective: {industry_findings[:120]}...\n\n"
        "Conclusion: Both academic evidence and market data confirm strong adoption momentum. "
        "The technology is mature enough for production deployment with appropriate governance."
    )

print("Specialist tools defined")

Specialist tools defined


In [16]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

if not SUPERVISOR_AVAILABLE:
    print("Capstone demo requires langgraph-supervisor")
    print("  uv pip install langgraph-supervisor>=0.1.0")
    print()
    print("Showing custom-coordinator fallback instead...")

    # ── Fallback: custom coordinator using Section 3 pattern ──────────────────
    from langgraph.graph import StateGraph, START, END
    from langgraph.types import Command
    from langchain_core.messages import SystemMessage, HumanMessage
    from langchain_openai import AzureChatOpenAI
    import operator
    from typing import Annotated
    from typing_extensions import TypedDict

    class ProductionState(TypedDict):
        topic: str
        technical: Annotated[list[str], operator.add]
        industry: Annotated[list[str], operator.add]
        synthesis: str
        phase: str

    llm = AzureChatOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
        temperature=0.7,
    )

    def tech_researcher(state: ProductionState) -> dict:
        result = search_technical_papers.invoke({"query": state["topic"]})
        print(f"[Tech Researcher] {result[:80]}...")
        return {"technical": [result], "phase": "industry_research"}

    def industry_analyst_node(state: ProductionState) -> dict:
        result = search_industry_reports.invoke({"query": state["topic"]})
        print(f"[Industry Analyst] {result[:80]}...")
        return {"industry": [result], "phase": "synthesis"}

    def synthesizer_node(state: ProductionState) -> dict:
        result = synthesize_research.invoke({
            "technical_findings": "\n".join(state["technical"]),
            "industry_findings": "\n".join(state["industry"]),
            "topic": state["topic"],
        })
        print(f"[Synthesizer] Report generated")
        return {"synthesis": result, "phase": "complete"}

    def router(state: ProductionState) -> str:
        return state["phase"]
    
    gb = StateGraph(ProductionState)
    gb.add_node("tech_research", tech_researcher)
    gb.add_node("industry_research", industry_analyst_node)
    gb.add_node("synthesis", synthesizer_node)
    gb.add_edge(START, "tech_research")
    gb.add_edge("tech_research", "industry_research")
    gb.add_edge("industry_research", "synthesis")
    gb.add_edge("synthesis", END)

    db = sqlite3.connect(":memory:", check_same_thread=False)
    memory = SqliteSaver(db)
    production_app = gb.compile(checkpointer=memory)

    config = {"configurable": {"thread_id": "production-fallback-001"}}
    result = production_app.invoke({
        "topic": "LangGraph adoption in enterprise AI systems",
        "technical": [],
        "industry": [],
        "synthesis": "",
        "phase": "tech_research",
    }, config)

    print()
    print("=" * 60)
    print(result["synthesis"])


else:
    from langgraph_supervisor import create_supervisor
    from langgraph.prebuilt import create_react_agent

    llm_fast = AzureChatOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        azure_deployment="gpt-4o-mini",
        temperature=0.7,
    )

    llm_smart  = AzureChatOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        azure_deployment="gpt-4o",
        temperature=0.7,
    )

    # ── Specialist agents ─────────────────────────────────────────────────────
    tech_researcher = create_react_agent(
        llm_fast,
        tools=[search_technical_papers],
        prompt="You are a technical research specialist. Search for academic papers and return structured findings.",
        name="technical_researcher",
    )

    industry_analyst = create_react_agent(
        llm_fast,
        tools=[search_industry_reports],
        prompt="You are an industry analyst. Search for market data and business insights.",
        name="industry_analyst",
    )

    synthesis_expert = create_react_agent(
        llm_smart,
        tools=[synthesize_research],
        prompt=(
            "You are a research synthesizer. Combine technical and industry findings. "
            "Always call synthesize_research with both technical_findings and industry_findings."
        ),
        name="synthesis_agent",
    )

    # ── Production supervisor with SQLite persistence ─────────────────────────
    db = sqlite3.connect("enterprise_research.db", check_same_thread=False)
    memory = SqliteSaver(db)

    research_system = create_supervisor(
        agents=[tech_researcher, industry_analyst, synthesis_expert],
        model=llm_smart,
        prompt=(
            "You coordinate a three-person research team:\n"
            "  - technical_researcher: for academic papers and technical facts\n"
            "  - industry_analyst: for market data and business insights\n"
            "  - synthesis_agent: for combining findings into a final analysis\n\n"
            "Workflow: always run technical_researcher first, then industry_analyst, "
            "then synthesis_agent. After synthesis, provide a final executive response "
            "to the user."
        ),
    ).compile(checkpointer=memory)

    config = {"configurable": {"thread_id": "production-research-001"}}

    print("=" * 60)
    print("Capstone: Production Multi-Agent Research System")
    print("=" * 60)

    result = research_system.invoke(
        {"messages": [{"role": "user", "content": (
            "Research the current state of LangGraph adoption in enterprise AI. "
            "Gather both technical and industry perspectives, then synthesize the findings."
        )}]},
        config,
    )

    print("\nFinal synthesis:")
    print("-" * 60)
    print(result["messages"][-1].content)

    # ── Resume same session (demonstrates persistence) ────────────────────────
    print()
    print("=" * 60)
    print("Follow-up question (same thread — state persists)")
    print("=" * 60)

    result2 = research_system.invoke(
        {"messages": [{"role": "user", "content": 
            "Based on your research, what are the top 2 risks for enterprise adoption?"}]},
        config,  # Same thread_id — supervisor remembers previous messages
    )

    print("\nFollow-up response:")
    print(result2["messages"][-1].content)


C:\Users\mohammed.a\AppData\Local\Temp\ipykernel_42096\1473927295.py:104: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  tech_researcher = create_react_agent(
C:\Users\mohammed.a\AppData\Local\Temp\ipykernel_42096\1473927295.py:111: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  industry_analyst = create_react_agent(
C:\Users\mohammed.a\AppData\Local\Temp\ipykernel_42096\1473927295.py:118: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  synthesis_expert = create_react_agent(


Capstone: Production Multi-Agent Research System

Final synthesis:
------------------------------------------------------------
### Executive Summary on LangGraph Adoption in Enterprise AI

LangGraph is evolving as a critical technology in the enterprise AI landscape, driven by recent technical advancements and increasing industry adoption. Research highlights its contributions to improving the effectiveness of agent-based systems through innovations like supervisor patterns and stateful agents. These enhancements yield significant performance gains, including reduced latency and improved task completion rates.

On the industry side, organizations adopting LangGraph report improved efficiency and responsiveness, providing a competitive edge. However, challenges such as integration with legacy systems and the need for specialized training remain barriers to entry.

In conclusion, LangGraph's structured, agent-centric framework is well-suited for tackling complex enterprise challenges, a